# Exercise 2.3: GCN and GraphSAGE on the Warsaw Bike-Sharing Graph

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/exercise_2_3_warsaw_gcn_graphsage_spatial_baselines.ipynb)

This notebook compares feature-only models with graph neural networks on bike-sharing station demand.

- Data source: Warsaw Bike-Sharing Daily Periods Graph Dataset for GNN, Season 2023, Mendeley Data, DOI: 10.17632/kzvdgfzk4w.1.
- Task: predict station-level trip intensity from station, weather, time, and spatial graph context.
- Models: Random Forest, XGBoost, GCN, GraphSAGE, and GraphSAGE with shuffled edges.
- Colab runtime: choose `Runtime` -> `Change runtime type` -> `T4 GPU`; otherwise neural network training can be slow.
- Main question: when does the graph add useful spatial information beyond ordinary tabular features?

## 1. Setup

The GNN models use PyTorch Geometric. The tabular baselines use scikit-learn and XGBoost.

In [ ]:
# Colab setup. In a local environment, run this only if a package is missing.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install networkx scikit-learn xgboost matplotlib pandas folium torch-geometric

In [ ]:
from pathlib import Path
from datetime import datetime
import fnmatch
import math
import pickle
import random
import urllib.request
import warnings
import zipfile

import folium
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, SAGEConv

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## 2. Load Warsaw graph snapshots

Use real `.pt` graph snapshots from the Warsaw dataset. The notebook downloads the course copy from Seafile and streams graph files from the archive, so Colab does not need to keep every NetworkX graph in RAM at the same time.

Each graph file is about 1.3 MB and contains roughly 295 stations. The default setting uses about one third of the graph snapshots, evenly sampled across the full date range, so the notebook remains practical in Colab.

The archive is about 340 MB. Set `DATASET_FRACTION = 1.0` for the full dataset, or set `MAX_GRAPHS` to a fixed number for a faster test run. Increase `RF_TREES`, `XGB_MAX_TREES`, and `GNN_EPOCHS` only when you want a slower but more stable comparison.

In [ ]:
GRAPH_PATTERN = "*.pt"
DATASET_FRACTION = 1 / 3
MAX_GRAPHS = None
RF_TREES = 60
XGB_MAX_TREES = 300
XGB_EARLY_STOPPING_ROUNDS = 25
HGB_ITERATIONS = 100
ABLATION_HGB_ITERATIONS = 60
GNN_EPOCHS = 120
GNN_PATIENCE = 20
DAILY_PERIOD_ORDER = ["morning_peak", "midday", "afternoon_peak", "evening", "night"]
DATA_DIR = Path("/content/warsaw_gnn") if IN_COLAB else Path("data/warsaw_gnn")
ARCHIVE_FILE = DATA_DIR / "warsaw_gnn_dataset.zip"
SEAFILE_ARCHIVE_URL = "https://seafile.rlp.net/seafhttp/f/d0d006cf506b4ed79da8/?op=view"
EXAMPLE_GRAPH_FILENAME = "afternoon_peak_01_06_2023.pt"
EXAMPLE_GRAPH_SIZE_BYTES = 1_319_004

def ensure_archive() -> Path:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE_FILE.exists():
        print(f"Downloading Warsaw GNN archive from Seafile to {ARCHIVE_FILE}...")
        urllib.request.urlretrieve(SEAFILE_ARCHIVE_URL, ARCHIVE_FILE)
    else:
        print(f"Using cached archive: {ARCHIVE_FILE}")
    return ARCHIVE_FILE

def graph_date_from_name(name: str) -> pd.Timestamp:
    parts = Path(name).stem.split("_")
    day, month, year = parts[-3:]
    return pd.Timestamp(datetime(int(year), int(month), int(day)))

def graph_period_from_name(name: str) -> str:
    parts = Path(name).stem.split("_")
    return "_".join(parts[:-3])

def evenly_sample(names: list[str], max_graphs: int | None) -> list[str]:
    if max_graphs is None or max_graphs >= len(names):
        return names
    positions = np.linspace(0, len(names) - 1, max_graphs).round().astype(int)
    return [names[i] for i in sorted(set(positions))]

def selected_graph_count(total_graphs: int, max_graphs: int | None = MAX_GRAPHS, fraction: float = DATASET_FRACTION) -> int:
    if max_graphs is not None:
        return min(max_graphs, total_graphs)
    if fraction >= 1.0:
        return total_graphs
    return max(1, int(round(total_graphs * fraction)))

def graph_names_from_archive(archive_file: Path, pattern: str = GRAPH_PATTERN, max_graphs: int | None = MAX_GRAPHS, fraction: float = DATASET_FRACTION) -> tuple[list[str], pd.DataFrame, pd.DataFrame]:
    with zipfile.ZipFile(archive_file) as zf:
        all_pt_names = sorted(
            [name for name in zf.namelist() if name.endswith(".pt")],
            key=graph_date_from_name,
        )
        matches = [name for name in all_pt_names if fnmatch.fnmatch(Path(name).name, pattern)]
        if not matches:
            raise FileNotFoundError(f"No graph files matching {pattern!r} were found in {archive_file}")

    selected_count = selected_graph_count(len(matches), max_graphs=max_graphs, fraction=fraction)
    selected = evenly_sample(matches, selected_count)
    all_dates = pd.Series([graph_date_from_name(name) for name in all_pt_names])
    match_dates = pd.Series([graph_date_from_name(name) for name in matches])
    selected_dates = pd.Series([graph_date_from_name(name) for name in selected])
    all_span_days = (all_dates.max() - all_dates.min()).days + 1
    selected_span_days = (selected_dates.max() - selected_dates.min()).days + 1

    summary = pd.DataFrame(
        [
            {
                "scope": "all graph files in archive",
                "graph_snapshots": len(all_pt_names),
                "periods": ", ".join(sorted({graph_period_from_name(name) for name in all_pt_names})),
                "date_start": all_dates.min().date(),
                "date_end": all_dates.max().date(),
                "time_span_days": all_span_days,
                "share_of_all_snapshots": 1.0,
                "share_of_all_time_span": 1.0,
            },
            {
                "scope": f"available files matching {pattern}",
                "graph_snapshots": len(matches),
                "periods": ", ".join(sorted({graph_period_from_name(name) for name in matches})),
                "date_start": match_dates.min().date(),
                "date_end": match_dates.max().date(),
                "time_span_days": (match_dates.max() - match_dates.min()).days + 1,
                "share_of_all_snapshots": len(matches) / len(all_pt_names),
                "share_of_all_time_span": ((match_dates.max() - match_dates.min()).days + 1) / all_span_days,
            },
            {
                "scope": f"used in this notebook ({len(selected)}/{len(matches)} matching files)",
                "graph_snapshots": len(selected),
                "periods": ", ".join(sorted({graph_period_from_name(name) for name in selected})),
                "date_start": selected_dates.min().date(),
                "date_end": selected_dates.max().date(),
                "time_span_days": selected_span_days,
                "share_of_all_snapshots": len(selected) / len(all_pt_names),
                "share_of_all_time_span": selected_span_days / all_span_days,
            },
        ]
    )
    selected_table = pd.DataFrame(
        {
            "graph_file": [Path(name).name for name in selected],
            "date": [graph_date_from_name(name).date() for name in selected],
            "period": [graph_period_from_name(name) for name in selected],
            "weekday": [graph_date_from_name(name).day_name() for name in selected],
        }
    )
    return selected, summary, selected_table

def prepare_archive_and_file_list() -> tuple[Path, list[str], pd.DataFrame, pd.DataFrame]:
    archive_file = ensure_archive()
    selected_names, coverage_summary, selected_graphs_table = graph_names_from_archive(archive_file)
    return archive_file, selected_names, coverage_summary, selected_graphs_table

archive_file, selected_names, coverage_summary, selected_graphs_table = prepare_archive_and_file_list()
observed_daily_periods = sorted({graph_period_from_name(name) for name in selected_names})
missing_daily_periods = [period for period in observed_daily_periods if period not in DAILY_PERIOD_ORDER]
if missing_daily_periods:
    DAILY_PERIOD_ORDER = [period for period in DAILY_PERIOD_ORDER if period in observed_daily_periods] + missing_daily_periods

display(coverage_summary)
display(selected_graphs_table.head(10))
display(selected_graphs_table.tail(10))
print(f"selected_graph_files={len(selected_names):,}")
print(f"daily_period_order={DAILY_PERIOD_ORDER}")

## 3. Convert graph attributes into a node table

The target is station-level trip intensity: incoming plus outgoing `trips_count`, transformed with `log1p`. Absolute coordinates such as `lat`, `lng`, `centroid_latitude`, and `centroid_longitude` are excluded from model features; they are used only for mapping and graph construction. Relative spatial descriptors such as `d_city_cen` and `d_metro_st` are kept as valid node features.

In [ ]:
def edge_weight(edge_data: dict) -> float:
    for key in ["trips_count", "trip_count", "count", "weight"]:
        if key in edge_data:
            try:
                return float(edge_data[key])
            except Exception:
                return 0.0
    return 1.0

def temporal_features_from_graph_file(graph_file: str) -> dict:
    graph_date = graph_date_from_name(graph_file)
    period = graph_period_from_name(graph_file)
    period_to_index = {name: idx for idx, name in enumerate(DAILY_PERIOD_ORDER)}
    if period not in period_to_index:
        raise ValueError(f"Unknown daily period {period!r}. Check the graph filename and update DAILY_PERIOD_ORDER.")

    period_index = period_to_index[period]
    weekday_index = graph_date.weekday()
    day_of_year = graph_date.dayofyear
    return {
        "graph_date": graph_date.date(),
        "daily_period": period,
        "time_sin": math.sin(2 * math.pi * period_index / len(DAILY_PERIOD_ORDER)),
        "time_cos": math.cos(2 * math.pi * period_index / len(DAILY_PERIOD_ORDER)),
        "weekday_sin": math.sin(2 * math.pi * weekday_index / 7),
        "weekday_cos": math.cos(2 * math.pi * weekday_index / 7),
        "year_sin": math.sin(2 * math.pi * day_of_year / 365),
        "year_cos": math.cos(2 * math.pi * day_of_year / 365),
    }

def graph_to_node_frame(graph: nx.DiGraph, graph_file: str) -> pd.DataFrame:
    graph_temporal_features = temporal_features_from_graph_file(graph_file)
    total_flow = {node: 0.0 for node in graph.nodes}
    in_flow = {node: 0.0 for node in graph.nodes}
    out_flow = {node: 0.0 for node in graph.nodes}
    for u, v, data in graph.edges(data=True):
        w = edge_weight(data)
        out_flow[u] += w
        in_flow[v] += w
        total_flow[u] += w
        total_flow[v] += w

    rows = []
    for node, attrs in graph.nodes(data=True):
        row = {"graph_file": graph_file, "node_id": node, **graph_temporal_features, **dict(attrs)}
        row["target_total_trips"] = total_flow[node]
        row["target_in_trips"] = in_flow[node]
        row["target_out_trips"] = out_flow[node]
        rows.append(row)
    return pd.DataFrame(rows)

def process_graphs_streaming(archive_file: Path, selected_names: list[str]) -> tuple[pd.DataFrame, list[torch.Tensor], pd.DataFrame, str, nx.DiGraph, pd.DataFrame]:
    node_frames = []
    graph_edge_indices = []
    graph_slices = []
    map_graph_file = ""
    map_graph = None
    edges_sample = pd.DataFrame()
    node_offset = 0

    with zipfile.ZipFile(archive_file) as zf:
        for graph_number, archive_name in enumerate(selected_names, start=1):
            graph_file = Path(archive_name).name
            with zf.open(archive_name) as f:
                graph = pickle.load(f)
            if not isinstance(graph, nx.Graph):
                raise TypeError(f"Expected a NetworkX graph in {graph_file}, got {type(graph)!r}")
            graph = nx.DiGraph(graph)

            node_frame = graph_to_node_frame(graph, graph_file)
            node_frames.append(node_frame)

            local_node_to_pos = {node: idx for idx, node in enumerate(graph.nodes())}
            src = []
            dst = []
            for u, v in graph.edges():
                if u in local_node_to_pos and v in local_node_to_pos and u != v:
                    src_pos = local_node_to_pos[u]
                    dst_pos = local_node_to_pos[v]
                    src.extend([src_pos, dst_pos])
                    dst.extend([dst_pos, src_pos])
            if not src:
                raise ValueError(f"No usable edges were found in {graph_file} after node alignment.")
            edge_index = torch.from_numpy(np.vstack([np.asarray(src, dtype=np.int64), np.asarray(dst, dtype=np.int64)])).long().contiguous()
            graph_edge_indices.append(edge_index)
            graph_slices.append(
                {
                    "graph_file": graph_file,
                    "row_start": node_offset,
                    "row_end": node_offset + len(node_frame),
                    "nodes": len(node_frame),
                    "edge_index_columns": int(edge_index.shape[1]),
                }
            )

            if map_graph is None:
                map_graph_file = graph_file
                map_graph = graph
                edges_sample = pd.DataFrame(
                    [{"graph_file": graph_file, "source": u, "target": v, "trips_count": edge_weight(data)} for u, v, data in graph.edges(data=True)]
                )
            else:
                del graph

            node_offset += len(node_frame)
            if graph_number % 100 == 0 or graph_number == len(selected_names):
                print(f"processed_graphs={graph_number:,}/{len(selected_names):,}, node_rows={node_offset:,}")

    if not node_frames:
        raise ValueError("No graph files were processed.")
    if not graph_edge_indices:
        raise ValueError("No usable edges were found after node alignment.")

    nodes = pd.concat(node_frames, ignore_index=True)
    graph_slices = pd.DataFrame(graph_slices)
    return nodes, graph_edge_indices, graph_slices, map_graph_file, map_graph, edges_sample

nodes, graph_edge_indices, graph_slices, map_graph_file, map_graph, edges_sample = process_graphs_streaming(archive_file, selected_names)
total_edge_index_edges = int(graph_slices["edge_index_columns"].sum())
nodes["y_log_total_trips"] = np.log1p(nodes["target_total_trips"].astype(float))

def is_absolute_coordinate_feature(column: str) -> bool:
    lower = column.lower()
    exact_coordinate_names = {"lat", "lng", "lon", "latitude", "longitude", "x", "y"}
    return lower in exact_coordinate_names or "centroid_latitude" in lower or "centroid_longitude" in lower

exclude_exact = {"graph_file", "graph_date", "daily_period", "node_id", "target_total_trips", "target_in_trips", "target_out_trips", "y_log_total_trips"}
exclude_contains = ["trip", "flow", "target"]
candidate_features = []
excluded_coordinate_features = []
for column in nodes.columns:
    if column in exclude_exact or any(term in column.lower() for term in exclude_contains):
        continue
    if is_absolute_coordinate_feature(column):
        excluded_coordinate_features.append(column)
        continue
    numeric = pd.to_numeric(nodes[column], errors="coerce")
    if numeric.notna().sum() >= max(5, int(0.2 * len(nodes))) and numeric.nunique(dropna=True) > 1:
        nodes[column] = numeric
        candidate_features.append(column)

if not candidate_features:
    raise ValueError("No numeric node features were found. Inspect the node attributes and select features manually.")

feature_frame = nodes[candidate_features].copy()
feature_frame = feature_frame.replace([np.inf, -np.inf], np.nan)
feature_frame = feature_frame.fillna(feature_frame.median(numeric_only=True)).fillna(0)

print(f"Using {len(candidate_features)} node features across {nodes['graph_file'].nunique()} graph snapshots:")
print(candidate_features)
temporal_feature_columns = ["time_sin", "time_cos", "weekday_sin", "weekday_cos", "year_sin", "year_cos"]
print("Cyclic temporal features used as model inputs:")
print([c for c in temporal_feature_columns if c in candidate_features])
print("Excluded absolute coordinate features:")
print(excluded_coordinate_features)
nodes[["graph_file", "node_id", "target_total_trips", "y_log_total_trips"] + candidate_features[:8]].head()

In [ ]:
display(edges_sample.head())
print(f"total_edge_index_columns={total_edge_index_edges:,}; displayed edge sample comes from {map_graph_file} with {len(edges_sample):,} directed edges")

fig, ax = plt.subplots(figsize=(7, 4))
nodes["target_total_trips"].hist(ax=ax, bins=30, color="#3f7f93", edgecolor="white")
ax.set_title("Station trip intensity distribution")
ax.set_xlabel("incoming + outgoing trips")
ax.set_ylabel("stations")
plt.show()

## 4. Train/test split and feature-only baselines

Random Forest and XGBoost use the same node table but no graph edges. This is the ordinary tabular baseline.

In [ ]:
X = feature_frame.to_numpy(dtype=np.float32)
y = nodes["y_log_total_trips"].to_numpy(dtype=np.float32)

graph_files = np.array(sorted(nodes["graph_file"].unique()))
train_graphs, test_graphs = train_test_split(
    graph_files, test_size=0.25, random_state=SEED
)
train_graphs, val_graphs = train_test_split(
    train_graphs, test_size=0.25, random_state=SEED
)
train_idx = nodes.index[nodes["graph_file"].isin(train_graphs)].to_numpy()
val_idx = nodes.index[nodes["graph_file"].isin(val_graphs)].to_numpy()
test_idx = nodes.index[nodes["graph_file"].isin(test_graphs)].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit(X[train_idx]).transform(X).astype(np.float32)
y_mean = float(y[train_idx].mean())
y_std = float(y[train_idx].std() + 1e-8)
print(f"train_graphs={len(train_graphs)}, validation_graphs={len(val_graphs)}, test_graphs={len(test_graphs)}")
print(f"train_rows={len(train_idx)}, validation_rows={len(val_idx)}, test_rows={len(test_idx)}")

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "rmse": float(math.sqrt(mse)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

results = []
predictions = {}

rf = RandomForestRegressor(n_estimators=RF_TREES, min_samples_leaf=5, max_samples=0.75, random_state=SEED, n_jobs=2)
rf.fit(X_scaled[train_idx], y[train_idx])
pred_rf = rf.predict(X_scaled[test_idx])
results.append({"model": "Random Forest", **regression_metrics(y[test_idx], pred_rf)})
predictions["Random Forest"] = pred_rf

try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=XGB_MAX_TREES,
        max_depth=4,
        learning_rate=0.04,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        tree_method="hist",
        early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS,
        n_jobs=2,
        random_state=SEED,
    )
    xgb.fit(
        X_scaled[train_idx],
        y[train_idx],
        eval_set=[(X_scaled[val_idx], y[val_idx])],
        verbose=False,
    )
    best_iteration = getattr(xgb, "best_iteration", None)
    if best_iteration is not None:
        print(f"XGBoost best_iteration={best_iteration + 1} / {XGB_MAX_TREES}")
    pred_xgb = xgb.predict(X_scaled[test_idx])
    results.append({"model": "XGBoost", **regression_metrics(y[test_idx], pred_xgb)})
    predictions["XGBoost"] = pred_xgb
except Exception as exc:
    print(f"XGBoost unavailable ({exc}). Using sklearn HistGradientBoostingRegressor as a fallback.")
    hgb = HistGradientBoostingRegressor(max_iter=HGB_ITERATIONS, learning_rate=0.05, random_state=SEED)
    hgb.fit(X_scaled[train_idx], y[train_idx])
    pred_hgb = hgb.predict(X_scaled[test_idx])
    results.append({"model": "HistGradientBoosting", **regression_metrics(y[test_idx], pred_hgb)})
    predictions["HistGradientBoosting"] = pred_hgb

pd.DataFrame(results).sort_values("rmse")

## 5. Build PyTorch Geometric graph tensors

The GNNs receive one PyTorch Geometric `Data` object per graph snapshot. A `DataLoader` batches these small graphs during training, which is easier for Colab RAM than one full-batch graph containing every snapshot. We also create shuffled-edge graphs as a negative control.

In [ ]:
BATCH_SIZE = 64

y_scaled = ((y - y_mean) / y_std).astype(np.float32)
split_by_graph = {graph_file: "unused" for graph_file in graph_slices["graph_file"]}
split_by_graph.update({graph_file: "train" for graph_file in train_graphs})
split_by_graph.update({graph_file: "val" for graph_file in val_graphs})
split_by_graph.update({graph_file: "test" for graph_file in test_graphs})

def make_snapshot_data(row: pd.Series, edge_index: torch.Tensor, shuffled: bool = False) -> Data:
    start = int(row["row_start"])
    end = int(row["row_end"])
    x_graph = torch.tensor(X_scaled[start:end], dtype=torch.float32)
    y_graph = torch.tensor(y_scaled[start:end].reshape(-1, 1), dtype=torch.float32)
    if shuffled:
        rng = np.random.default_rng(SEED + start)
        node_permutation = torch.from_numpy(rng.permutation(end - start)).long()
        edge_index = node_permutation[edge_index]
    data = Data(x=x_graph, edge_index=edge_index, y=y_graph)
    data.graph_file = row["graph_file"]
    return data

graph_data = []
graph_data_shuffled = []
for row, edge_index in zip(graph_slices.itertuples(index=False), graph_edge_indices):
    row_series = pd.Series(row._asdict())
    graph_data.append(make_snapshot_data(row_series, edge_index, shuffled=False))
    graph_data_shuffled.append(make_snapshot_data(row_series, edge_index, shuffled=True))

train_data = [data for data in graph_data if split_by_graph[data.graph_file] == "train"]
val_data = [data for data in graph_data if split_by_graph[data.graph_file] == "val"]
test_data = [data for data in graph_data if split_by_graph[data.graph_file] == "test"]
train_data_shuffled = [data for data in graph_data_shuffled if split_by_graph[data.graph_file] == "train"]
val_data_shuffled = [data for data in graph_data_shuffled if split_by_graph[data.graph_file] == "val"]
test_data_shuffled = [data for data in graph_data_shuffled if split_by_graph[data.graph_file] == "test"]

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
train_loader_shuffled = DataLoader(train_data_shuffled, batch_size=BATCH_SIZE, shuffle=True)
val_loader_shuffled = DataLoader(val_data_shuffled, batch_size=BATCH_SIZE, shuffle=False)
test_loader_shuffled = DataLoader(test_data_shuffled, batch_size=BATCH_SIZE, shuffle=False)

print(f"snapshot graphs: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}, batch_size={BATCH_SIZE}")
print(f"total edge_index columns={total_edge_index_edges:,}; mini-batch training keeps only one batch on {DEVICE} at a time")

## 6. GCN and GraphSAGE with PyTorch Geometric

- GCN uses `GCNConv`.
- GraphSAGE uses `SAGEConv`.
- The shuffled-edge GraphSAGE model has the same node features and a graph of similar size, but the spatial relations are wrong.

In [ ]:
class PyGGCNRegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.out = torch.nn.Linear(hidden_dim, 1)
        self.dropout = torch.nn.Dropout(0.15)

    def forward(self, data: Data) -> torch.Tensor:
        h = torch.relu(self.conv1(data.x, data.edge_index))
        h = self.dropout(h)
        h = torch.relu(self.conv2(h, data.edge_index))
        return self.out(h)

class PyGGraphSAGERegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.out = torch.nn.Linear(hidden_dim, 1)
        self.dropout = torch.nn.Dropout(0.15)

    def forward(self, data: Data) -> torch.Tensor:
        h = torch.relu(self.conv1(data.x, data.edge_index))
        h = self.dropout(h)
        h = torch.relu(self.conv2(h, data.edge_index))
        return self.out(h)

def loader_loss(model: torch.nn.Module, loader: DataLoader, loss_fn: torch.nn.Module) -> float:
    model.eval()
    total_loss = 0.0
    total_nodes = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred = model(batch)
            loss = loss_fn(pred, batch.y)
            total_loss += float(loss.detach().cpu()) * batch.num_nodes
            total_nodes += batch.num_nodes
    return total_loss / max(total_nodes, 1)

def predict_loader(model: torch.nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred_scaled = model(batch).detach().cpu().numpy().ravel()
            preds.append(pred_scaled * y_std + y_mean)
    return np.concatenate(preds)

def train_gnn(model: torch.nn.Module, train_loader: DataLoader, val_loader: DataLoader, test_loader: DataLoader, model_name: str, epochs: int = GNN_EPOCHS, lr: float = 0.01, patience: int = GNN_PATIENCE, log_every: int = 25) -> tuple[np.ndarray, pd.DataFrame]:
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = torch.nn.SmoothL1Loss()
    history = []
    best_state = None
    best_val_loss = float("inf")
    stale_epochs = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_nodes = 0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            pred = model(batch)
            loss = loss_fn(pred, batch.y)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.detach().cpu()) * batch.num_nodes
            running_nodes += batch.num_nodes

        train_loss = running_loss / max(running_nodes, 1)
        val_loss = loader_loss(model, val_loader, loss_fn)
        row = {"model": model_name, "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss}
        history.append(row)
        if epoch == 0 or (epoch + 1) % log_every == 0:
            print(f"{model_name:26s} epoch={epoch + 1:03d} train_loss={row['train_loss']:.4f} val_loss={row['val_loss']:.4f}")

        if row["val_loss"] < best_val_loss - 1e-5:
            best_val_loss = row["val_loss"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= patience:
            break
    print(f"{model_name:26s} stopped_after={len(history)} best_val_loss={best_val_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return predict_loader(model, test_loader), pd.DataFrame(history)

torch.manual_seed(SEED)
pred_gcn, hist_gcn = train_gnn(PyGGCNRegressor(X_scaled.shape[1]), train_loader, val_loader, test_loader, "GCN")
results.append({"model": "GCN", **regression_metrics(y[test_idx], pred_gcn)})
predictions["GCN"] = pred_gcn

torch.manual_seed(SEED)
pred_sage, hist_sage = train_gnn(PyGGraphSAGERegressor(X_scaled.shape[1]), train_loader, val_loader, test_loader, "GraphSAGE")
results.append({"model": "GraphSAGE", **regression_metrics(y[test_idx], pred_sage)})
predictions["GraphSAGE"] = pred_sage

torch.manual_seed(SEED)
pred_sage_shuffled, hist_sage_shuffled = train_gnn(PyGGraphSAGERegressor(X_scaled.shape[1]), train_loader_shuffled, val_loader_shuffled, test_loader_shuffled, "GraphSAGE shuffled edges")
results.append({"model": "GraphSAGE shuffled edges", **regression_metrics(y[test_idx], pred_sage_shuffled)})
predictions["GraphSAGE shuffled edges"] = pred_sage_shuffled
training_history = pd.concat([hist_gcn, hist_sage, hist_sage_shuffled], ignore_index=True)

pd.DataFrame(results).sort_values("rmse")

## 7. Compare results

Lower RMSE and MAE are better. Higher R2 is better.

In [ ]:
metrics_table = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(metrics_table)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, color in zip(axes, ["rmse", "mae", "r2"], ["#507dbc", "#3f7f93", "#8a6f3d"]):
    ordered = metrics_table.sort_values(metric, ascending=(metric != "r2"))
    ax.barh(ordered["model"], ordered[metric], color=color)
    ax.set_title(metric.upper())
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
for model_name, model_history in training_history.groupby("model"):
    ax.plot(model_history["epoch"], model_history["val_loss"], label=model_name)
ax.set_title("GNN validation loss during training")
ax.set_xlabel("epoch")
ax.set_ylabel("SmoothL1 loss on standardized target")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

display(training_history.groupby("model").tail(1).reset_index(drop=True))

In [ ]:
best_model = metrics_table.iloc[0]["model"]
best_pred = predictions[best_model]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y[test_idx], best_pred, alpha=0.75, color="#2f6f73", edgecolor="white", linewidth=0.5)
lims = [min(y[test_idx].min(), best_pred.min()), max(y[test_idx].max(), best_pred.max())]
ax.plot(lims, lims, color="#333333", linestyle="--", linewidth=1)
ax.set_xlabel("observed log1p trips")
ax.set_ylabel("predicted log1p trips")
ax.set_title(f"Observed vs predicted: {best_model}")
ax.grid(alpha=0.25)
plt.show()

## 8. Spatial diagnostics

A strong tree model can win because the node table contains useful relative spatial descriptors such as distances to the city center, metro stations, and other points of interest. Absolute coordinates are not used as model features. Use these diagnostics to separate two questions:

- Does the real graph carry signal? Compare GraphSAGE with real edges against GraphSAGE with shuffled edges.
- How much do relative distance features help ordinary feature-only models? Remove only the relative distance variables and compare a fast gradient-boosting diagnostic model.

In [ ]:
score_by_model = metrics_table.set_index("model")
diagnostics = []

if {"GraphSAGE", "GraphSAGE shuffled edges"}.issubset(score_by_model.index):
    real_rmse = score_by_model.loc["GraphSAGE", "rmse"]
    shuffled_rmse = score_by_model.loc["GraphSAGE shuffled edges", "rmse"]
    diagnostics.append(
        {
            "question": "Do real edges matter?",
            "comparison": "GraphSAGE shuffled RMSE - GraphSAGE RMSE",
            "delta_rmse": shuffled_rmse - real_rmse,
            "interpretation": "positive means the real station graph helps",
        }
    )

feature_only = [m for m in ["Random Forest", "XGBoost", "HistGradientBoosting"] if m in score_by_model.index]
gnn_models = [m for m in ["GCN", "GraphSAGE"] if m in score_by_model.index]
if feature_only and gnn_models:
    best_feature_rmse = score_by_model.loc[feature_only, "rmse"].min()
    best_gnn_rmse = score_by_model.loc[gnn_models, "rmse"].min()
    diagnostics.append(
        {
            "question": "Does a GNN beat feature-only models here?",
            "comparison": "best feature-only RMSE - best GNN RMSE",
            "delta_rmse": best_feature_rmse - best_gnn_rmse,
            "interpretation": "positive means a GNN wins; negative means tabular features are stronger",
        }
    )

pd.DataFrame(diagnostics)

In [ ]:
def is_relative_distance_feature(column: str) -> bool:
    lower = column.lower()
    return lower.startswith("d_") or "distance" in lower or lower.endswith("_dist") or lower.startswith("dist_")

relative_distance_feature_columns = [c for c in candidate_features if is_relative_distance_feature(c)]
without_relative_distance_columns = [c for c in candidate_features if c not in relative_distance_feature_columns]

print(f"Absolute coordinate features excluded from all model features: {len(excluded_coordinate_features)}")
print(excluded_coordinate_features)
print(f"Relative distance features kept in default feature set: {len(relative_distance_feature_columns)}")
print(relative_distance_feature_columns[:30])
print(f"Features remaining after relative distance ablation: {len(without_relative_distance_columns)}")

def scaled_feature_matrix(feature_columns: list[str]) -> np.ndarray:
    frame = nodes[feature_columns].replace([np.inf, -np.inf], np.nan)
    frame = frame.fillna(frame.iloc[train_idx].median(numeric_only=True)).fillna(0)
    values = frame.to_numpy(dtype=np.float32)
    return StandardScaler().fit(values[train_idx]).transform(values).astype(np.float32)

def run_tabular_ablation(feature_columns: list[str], label: str) -> list[dict]:
    if not feature_columns:
        return []
    X_variant = scaled_feature_matrix(feature_columns)
    hgb_variant = HistGradientBoostingRegressor(max_iter=ABLATION_HGB_ITERATIONS, learning_rate=0.06, random_state=SEED)
    hgb_variant.fit(X_variant[train_idx], y[train_idx])
    return [{"feature_set": label, "model": "Fast HistGradientBoosting", **regression_metrics(y[test_idx], hgb_variant.predict(X_variant[test_idx]))}]

tabular_ablation = []
tabular_ablation.extend(run_tabular_ablation(candidate_features, "default features: no absolute coordinates"))
tabular_ablation.extend(run_tabular_ablation(without_relative_distance_columns, "without relative distance features"))
tabular_ablation_table = pd.DataFrame(tabular_ablation).sort_values(["feature_set", "rmse"])
display(tabular_ablation_table)

## 9. Folium graph map

The map shows station nodes by geographic position and draws the strongest directed bike-flow edges for one selected snapshot.

In [ ]:
lat_candidates = [c for c in nodes.columns if c.lower() in {"lat", "latitude", "y"} or "latitude" in c.lower()]
lng_candidates = [c for c in nodes.columns if c.lower() in {"lng", "lon", "longitude", "x"} or "longitude" in c.lower()]

if lat_candidates and lng_candidates:
    lat_col, lng_col = lat_candidates[0], lng_candidates[0]
    plot_nodes = nodes[nodes["graph_file"] == map_graph_file].copy()
    plot_nodes[lat_col] = pd.to_numeric(plot_nodes[lat_col], errors="coerce")
    plot_nodes[lng_col] = pd.to_numeric(plot_nodes[lng_col], errors="coerce")
    plot_nodes = plot_nodes.dropna(subset=[lat_col, lng_col])
    station_lookup = plot_nodes.set_index("node_id")[[lat_col, lng_col, "target_total_trips"]].to_dict("index")

    center = [plot_nodes[lat_col].mean(), plot_nodes[lng_col].mean()]
    m = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

    flow_edges = sorted(
        [(u, v, edge_weight(data)) for u, v, data in map_graph.edges(data=True)],
        key=lambda item: item[2],
        reverse=True,
    )[:500]
    max_flow = max([w for _, _, w in flow_edges], default=1.0)
    for u, v, w in flow_edges:
        if u not in station_lookup or v not in station_lookup:
            continue
        source = station_lookup[u]
        target = station_lookup[v]
        folium.PolyLine(
            locations=[(source[lat_col], source[lng_col]), (target[lat_col], target[lng_col])],
            color="#5068a9",
            weight=0.5 + 3.0 * (w / max_flow),
            opacity=0.22,
            tooltip=f"{u} -> {v}: trips_count={w:.0f}",
        ).add_to(m)

    max_target = plot_nodes["target_total_trips"].max()
    for _, row in plot_nodes.iterrows():
        intensity = row["target_total_trips"] / max_target if max_target else 0
        color = "#1f77b4" if intensity < 0.33 else "#2ca02c" if intensity < 0.66 else "#d62728"
        folium.CircleMarker(
            location=(row[lat_col], row[lng_col]),
            radius=3 + 5 * intensity,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            weight=1,
            tooltip=f"station={row['node_id']}<br>total_trips={row['target_total_trips']:.0f}",
        ).add_to(m)

    print(f"Folium map for {map_graph_file}: {len(plot_nodes)} stations, top {len(flow_edges)} directed flow edges")
    display(m)
else:
    print("No latitude/longitude columns found. Available columns:")
    print(list(nodes.columns))

## 10. Interpretation checklist

- Absolute coordinate columns are not model features in this notebook. They are used only for mapping and graph-related spatial context.
- If XGBoost or Random Forest wins on the default feature set, this does not mean space is irrelevant. Relative distance features such as `d_city_cen` and `d_metro_st` still encode interpretable urban structure.
- If GraphSAGE beats the shuffled-edge GraphSAGE model, the real station graph carries useful spatial or mobility structure.
- If tabular models get worse after removing relative distance variables, ordinary feature-only models were also using spatial information, but not absolute coordinate memorization.
- GCN can perform worse than GraphSAGE here because simple adjacency smoothing can over-smooth station features on a small, dense mobility graph.
- Results are more meaningful because train/validation/test splits are separated by graph snapshot, not by random nodes inside the same snapshot.

Exercise extension: change `GRAPH_PATTERN` to a single period such as `morning_peak_*.pt` or `afternoon_peak_*.pt` and test whether GNN gains differ by daily period.